In [2]:
import pandas as pd

In [3]:
df_train = pd.read_csv('Data_train.csv')
df_test = pd.read_csv('Data_test.csv')

FileNotFoundError: [Errno 2] No such file or directory: 'Data_train.csv'

In [ ]:
print(df_train.columns.tolist())
print(df_test.columns.tolist())
count=0

#сколько всего пропусков
for i in df_train.columns:
    if df_train[i].isnull().sum() > 0:
        count += 1

print(count)

['type', 'group', 'education', 'meal', 'preparation course', 'score-1', 'score-2', 'score-3']
['type', 'group', 'education', 'meal', 'preparation course']
1


In [ ]:
df_train.dtypes

,0
type,object
group,object
education,object
meal,object
preparation course,object
score-1,int64
score-2,int64
score-3,int64
Pass,int64


Заполните пропуски в столбце уникальной категорией, если столбец с пропуском категориальный, и средним значением, если столбец числовой. Заполняйте одновременно и df_train, и df_test - одинаковым образом. В ответе укажите количество различных значений, потребовавшихся для заполнения пропусков (это равно количеству новых уникальных категорий плюс количество средних значений для заполнения пропусков в числовых столбцах).

In [ ]:
cnt = 0
for i in df_train.columns:
    if i not in ['score-1', 'score-2', 'score-3']:
      if df_train[i].dtype == 'object':
          cnt += 1
          df_train[i] = df_train[i].fillna('Unknown')
          df_test[i] = df_test[i].fillna('Unknown')
      else:
          cnt += 1
          mean_val = df_train[i].mean()
          df_train[i] = df_train[i].fillna(mean_val)
          df_test[i] = df_test[i].fillna(mean_val)

Кошка прошла полосу препятствий по мнению судьи, если он поставил ей больше 50 баллов. Кошка считается прошедшей полосу препятствий, если все судьи поставили ей больше 50 баллов. В df_train создайте колонку 'Pass' и запишите в неё 1, если кошка прошла полосу препятствий, и 0 иначе. В ответ запишите, сколько кошек из df_train не прошли полосу препятствий.

В df_test от вас скрыта информация о судейских баллах, поэтому неизвестно, прошла кошка полосу препятствия или нет - это и надо будет предсказать в заданиях ниже.



In [ ]:
df_train['Pass'] = (
    (df_train['score-1'] > 50) &
    (df_train['score-2'] > 50) &
    (df_train['score-3'] > 50)
).astype(int)

print((df_train['Pass'] == 0).sum())

145


1) Среди всех диких кошек найдите долю кошек, прошедших полосу препятствий. Такую же долю рассчитайте для домашних кошек. В ответе укажите модуль разности этих долей. Ответ округлите до сотых.

In [ ]:
print(((df_train['type']=='wild') & (df_train['Pass'] == 1)).sum() / ((df_train['type']=='wild')).sum())
print(((df_train['type']=='domestic') & (df_train['Pass'] == 1)).sum() / ((df_train['type']=='domestic')).sum())
print(round(abs(((df_train['type']=='wild') & (df_train['Pass'] == 1)).sum() / ((df_train['type']=='wild')).sum() - ((df_train['type']=='domestic') & (df_train['Pass'] == 1)).sum() / ((df_train['type']=='domestic')).sum()), 2))

0.7813411078717201
0.803921568627451
0.02


2) Сколько кошек среди не прошедших полосу препятствий имели инструктора с уровнем образования "high school"?


In [ ]:
print(((df_train['education']=='high school') & (df_train['Pass'] == 0)).sum())


35


In [ ]:
df_train['preparation course']

,preparation course
0,completed
1,none
2,completed
3,none
4,none
...,...
695,completed
696,none
697,completed
698,none


3) Сколько диких кошек среди прошедших полосу препятствий не проходили специальный курс подготовки?

In [ ]:
print(((df_train['type']=='wild') & (df_train['preparation course'] =='none')).sum())


212


4) Чему равна медиана баллов, выставленных первым судьей?


In [ ]:
print(df_train['score-1'].median())

66.0


5) Найдите межквартильный размах баллов третьего судьи (третья квартиль минус первая квартиль) для домашних кошек, не проходивших специальный курс подготовки.

Комментарий: для вычисления квартилей дискретного распределения используйте интерполяцию меньшим значением (lower interpolation). Это означает, что если искомая квартиль лежит между двумя измерениями i и j, то значение квартили равно i.

In [ ]:
filtered = df_train[
    (df_train['type'] == 'domestic') &
    (df_train['preparation course'] == 'none')
]['score-3']

q1 = filtered.quantile(0.25, interpolation='lower')
q3 = filtered.quantile(0.75, interpolation='lower')

print(q3 - q1)

20


 Далее используйте только категориальные столбцы. Закодируйте их с помощью One-hot encoding с учетом того, что мы не хотим получить мультиколлинеарности в новых данных. Сколько получилось числовых столбцов из исходных категориальных? Кодируйте и df_train, и df_test.


In [ ]:
df_train = pd.get_dummies(df_train,
                          columns=['type', 'education', 'meal', 'preparation course'],
                          drop_first=True,
                          dtype=int)

df_test = pd.get_dummies(df_test,
                         columns=['type', 'education', 'meal', 'preparation course'],
                         drop_first=True,
                         dtype=int)

KeyError: "None of [Index(['type', 'education', 'meal', 'preparation course'], dtype='object')] are in the [columns]"

In [ ]:
cols_after = df_train.shape[1]
print(df_train.shape[1])

22


In [ ]:

y_train = df_train['Pass']
X_train = df_train.drop('Pass', axis=1)
X_train = df_train.drop(['Pass', 'score-1', 'score-2', 'score-3'], axis=1)


. Попытаемся по характеристикам кошки (бывшие категориальные, а теперь - числовые столбцы) предсказать, прошла она полосу препятствий или нет.


Сформируйте из df_train матрицу объект-признак X и вектор ответов y.


Обучите решающее дерево (DecisionTreeClassifier из библиотеки sklearn.tree) глубины 5 с энтропийным критерием информативности на закодированных в пункте а) тренировочных данных по кросс-валидации с тремя фолдами, метрика качества - roc-auc.


Чему равен roc-auc, усредненный по фолдам? Ответ округлите до десятых.


Комментарий: остальные гиперпараметры дерева оставьте дефолтными (splitter='best', min_samples_split=2, min_samples_leaf=1, min_weight_fraction_leaf=0.0, max_features=None, random_state=None, max_leaf_nodes=None, min_impurity_decrease=0.0, min_impurity_split=None, class_weight=None, ccp_alpha=0.0)

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import cross_val_score

dt = DecisionTreeClassifier(max_depth=5, criterion='entropy')

scores = cross_val_score(dt, X_train, y_train, cv=3, scoring='roc_auc')

print(round(scores.mean(), 1))

0.7


Подберите глубину решающего дерева (max_depth), перебирая глубину от 2 до 20 с шагом 1 и используя перебор по сетке (GridSearchCV из библиотеки sklearn.model_selection) с тремя фолдами и метрикой качества - roc-auc. В ответ запишите наилучшее среди искомых значение max_depth.


Комментарий: остальные гиперпараметры дерева оставьте дефолтными (splitter='best', min_samples_split=2, min_samples_leaf=1, min_weight_fraction_leaf=0.0, max_features=None, random_state=None, max_leaf_nodes=None, min_impurity_decrease=0.0, min_impurity_split=None, class_weight=None, ccp_alpha=0.0)

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import GridSearchCV

param_grid = {
    'max_depth': list(range(2, 21))
}

dt = DecisionTreeClassifier()

grid_search = GridSearchCV(estimator=dt, param_grid=param_grid, cv=3, scoring='roc_auc')
grid_search.fit(X_train, y_train)

print(grid_search.best_params_['max_depth'])

2


 Добавьте к данным новый признак cat_bio, содержащий в качестве значений пары значений из столбца type и столбца group. Например, если кошка имеет type='wild' и  group='group B', то в cat_bio будет стоять строка '(wild, group B)'. Примените OneHotEncoding (с учетом того, что мы не хотим получить мультиколлинеарности в новых данных) к столбцам 'cat_bio', 'education', 'meal', 'preparation course', а затем обучите решающее дерево глубины 5 с энтропийным критерием информативности на полученных после кодирования данных. Чему равен roc-auc? Ответ округлите до сотых.

Комментарий: остальные гиперпараметры дерева оставьте дефолтными (splitter='best', min_samples_split=2, min_samples_leaf=1, min_weight_fraction_leaf=0.0, max_features=None, random_state=None, max_leaf_nodes=None, min_impurity_decrease=0.0, min_impurity_split=None, class_weight=None, ccp_alpha=0.0)



In [ ]:
# Создаём новый признак cat_bio
df_train['cat_bio'] = df_train.apply(lambda row: f"({row['type']}, {row['group']})", axis=1)
df_test['cat_bio'] = df_test.apply(lambda row: f"({row['type']}, {row['group']})", axis=1)

# One-Hot Encoding
cols_to_encode = ['cat_bio', 'education', 'meal', 'preparation course']
df_train_encoded = pd.get_dummies(df_train, columns=cols_to_encode, drop_first=True, dtype=int)
df_test_encoded = pd.get_dummies(df_test, columns=cols_to_encode, drop_first=True, dtype=int)

# Формируем X и y
y_train = df_train_encoded['Pass']
X_train = df_train_encoded.drop(['Pass', 'score-1', 'score-2', 'score-3', 'type', 'group'], axis=1)

# Обучаем дерево с кросс-валидацией
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import cross_val_score

dt = DecisionTreeClassifier(max_depth=5, criterion='entropy')
scores = cross_val_score(dt, X_train, y_train, cv=3, scoring='roc_auc')

print(round(scores.mean(), 2))

Теперь вы можете использовать любую модель машинного обучения для решения задачи. Также можете делать любую другую обработку признаков. Ваша задача - получить наилучшее качество (ROC_AUC).

Качество проверяется на тестовых данных.

ROC_AUC > 0.7 - 0.25 балла
ROC_AUC > 0.74 - 0.75 балла

In [1]:
from catboost import CatBoostClassifier
from sklearn.model_selection import cross_val_score

# Можно работать с исходными данными без get_dummies!
# CatBoost сам обрабатывает категориальные признаки

# Перечитай данные заново если уже закодировал
# df_train = pd.read_csv('Data_train.csv') + заполни пропуски + создай Pass

cat_features = ['type', 'group', 'education', 'meal', 'preparation course']

X_train = df_train.drop(['Pass', 'score-1', 'score-2', 'score-3'], axis=1)
y_train = df_train['Pass']

model = CatBoostClassifier(
    cat_features=cat_features,
    verbose=0  # убирает лишний вывод
)

# model = CatBoostClassifier(
#     cat_features=cat_features,
#     iterations=500,
#     depth=6,
#     learning_rate=0.05,
#     verbose=0
# )


scores = cross_val_score(model, X_train, y_train, cv=3, scoring='roc_auc')
print(round(scores.mean(), 2))

ModuleNotFoundError: No module named 'catboost'